# Traducción de HumAID al español

Traduce los cuatro conjuntos de HumAID (train, val, test y test_intra) al español para la etapa 2. Primero compara tres traductores en un piloto de 1.000 tweets, después traduce todo con el elegido y revisa la calidad. Chile 2014 ya está en español y no se traduce.

## 1. Repositorio, dependencias y datos

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JuanTrujilloM/final-project-NN-DL.git"
BRANCH = "main"

# On Kaggle the repo is cloned; locally the notebook runs from the repo root.
if Path("/kaggle").exists():
    if not Path("final-project-NN-DL").exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL], check=True)
    os.chdir("final-project-NN-DL")
elif Path.cwd().name == "notebooks":
    os.chdir("..")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if not Path("data/processed/humaid_en/train.csv").exists():
    subprocess.run([sys.executable, "scripts/prepare_data.py"], check=True)
print("Directorio de trabajo:", Path.cwd())

## 2. Librerías y configuración

In [ ]:
import gc
import json
import random
import re
import shutil
import time

import emoji
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers.utils import logging as hf_logging

# Hides the repeated max_new_tokens notice that generate prints on every batch.
hf_logging.set_verbosity_error()

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


def set_all_seeds(seed):
    os.environ["PL_GLOBAL_SEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_all_seeds(SEED)

In [ ]:
DATA_DIR = Path("data/processed/humaid_en")
OUT_DIR = Path("data/raw/humaid_events_translated")
CACHE_DIR = OUT_DIR / "cache"
RESULTS_DIR = Path("results/traduccion")
for path in [OUT_DIR, CACHE_DIR, RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

TEXT_COL = "text_clean"
SPLITS = ["train", "val", "test", "test_intra"]

# None to translate all data
LIMIT = None

TRANSLATORS = {
    "opus_big": {
        "name": "opus_big",
        "model_name": "Helsinki-NLP/opus-mt-tc-big-en-es",
        "tokenizer_kwargs": {},
        "tgt_lang": None,
    },
    "opus_base": {
        "name": "opus_base",
        "model_name": "Helsinki-NLP/opus-mt-en-es",
        "tokenizer_kwargs": {},
        "tgt_lang": None,
    },
    "nllb": {
        "name": "nllb",
        "model_name": "facebook/nllb-200-distilled-600M",
        "tokenizer_kwargs": {"src_lang": "eng_Latn"},
        "tgt_lang": "spa_Latn",
    },
}

GENERATION = {
    "batch_size": 64,
    "num_beams": 4,
    "max_input_tokens": 256,
    "max_new_tokens": 256,
    "fp16": True,
}

PILOT_PER_CLASS = 100
CHUNK_SIZE = 5000

## 3. Datos

In [ ]:
def load_split(name, limit=LIMIT):
    df = pd.read_csv(DATA_DIR / f"{name}.csv")
    if limit is not None and limit < len(df):
        df = df.sample(n=limit, random_state=SEED)
    df = df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
    df["tweet_id"] = df["tweet_id"].astype(str)
    return df


data = {name: load_split(name) for name in SPLITS}
for name, df in data.items():
    print(f"{name}: {len(df):,} tweets, {df[TEXT_COL].str.len().sum():,} caracteres")
total_tweets = sum(len(df) for df in data.values())
print(f"Total: {total_tweets:,} tweets")

## 4. Preparación del texto

Las pruebas con tweets reales mostraron cuatro problemas del traductor:

- Borra la `@` de `@USER` y a veces escribe `User`. Después de traducir se restaura `@USER`.
- Un `@USER` al inicio a veces lo traduce como "los usuarios". Los `@USER` del inicio y del final se separan antes de traducir y se vuelven a poner después.
- Borra los emojis. Se sacan antes de traducir y se agregan al final del tweet.
- Con varias frases seguidas a veces omite la última. Cada tweet se parte en frases y se traducen por separado, sin cortar en abreviaturas como `U.S.` o `Dr.`.

Además, el 6,5 % de los tweets de HumAID trae emojis dañados desde el origen (`ὤF`, `ἿD`), que el traductor convierte en basura como `FDFDD`. Se quitan antes de traducir; el texto en inglés no cambia.

In [ ]:
BROKEN_EMOJI_RE = re.compile(r"[\u1f00-\u1fff][0-9A-F]?")
SENTENCE_RE = re.compile(r"(?<=[.!?…])\s+")
ABBREVIATION_RE = re.compile(r"(?:\b(?:[A-Z]\.){2,}|\b(?:Mr|Mrs|Ms|Dr|St|Ft|Mt|Jr|Sr|Gov|Sen|Rep|Gen|Lt|Sgt|No|vs|approx|est)\.)$")
LEADING_USERS_RE = re.compile(r"^(?:[.\s|:,]*@USER)+[\s|:,]*")
TRAILING_USERS_RE = re.compile(r"(?:[\s|:,]*@USER)+[\s|:,.]*$")
USER_OUT_RE = re.compile(r"@?\s?\bUSER\b", re.IGNORECASE)
SPACE_PUNCT_RE = re.compile(r"\s+([.,;:!?])")
SPACE_RE = re.compile(r"\s+")


# Splits on sentence ends, but joins back pieces that end in an abbreviation.
def split_sentences(text):
    sentences = []
    for piece in SENTENCE_RE.split(text):
        if sentences and ABBREVIATION_RE.search(sentences[-1]):
            sentences[-1] = f"{sentences[-1]} {piece}"
        elif piece:
            sentences.append(piece)
    return sentences


# Splits a tweet into its translatable sentences plus the parts that are kept as they are.
def split_tweet(text):
    emojis = "".join(item["emoji"] for item in emoji.emoji_list(text))
    body = BROKEN_EMOJI_RE.sub(" ", emoji.replace_emoji(text, ""))
    body = SPACE_RE.sub(" ", body).strip()
    lead = LEADING_USERS_RE.match(body)
    lead = lead.group() if lead else ""
    body = body[len(lead):]
    trail = TRAILING_USERS_RE.search(body)
    trail = trail.group() if trail else ""
    body = body[:len(body) - len(trail)]
    kept = {"lead": " ".join(["@USER"] * lead.count("@USER")),
            "trail": " ".join(["@USER"] * trail.count("@USER")),
            "emojis": emojis}
    return split_sentences(body.strip()), kept


def join_tweet(sentences, kept):
    text = USER_OUT_RE.sub(" @USER ", " ".join(sentences))
    text = " ".join([kept["lead"], text, kept["trail"], kept["emojis"]])
    return SPACE_PUNCT_RE.sub(r"\1", SPACE_RE.sub(" ", text)).strip()


example = data["train"].loc[data["train"][TEXT_COL].str.startswith("@USER"), TEXT_COL].iloc[0]
print(example)
print(split_tweet(example))

## 5. Traductor

In [ ]:
def load_translator(hp):
    tokenizer = AutoTokenizer.from_pretrained(hp["model_name"], **hp["tokenizer_kwargs"])
    model = AutoModelForSeq2SeqLM.from_pretrained(hp["model_name"]).to(DEVICE).eval()
    if GENERATION["fp16"] and DEVICE.type == "cuda":
        model = model.half()
    return tokenizer, model


# Translates sentences in batches sorted by length, so each batch has little padding.
# Also reports whether the output has more unknown tokens than the input; each one drops a character such as Ú.
@torch.inference_mode()
def translate_sentences(sentences, tokenizer, model, hp, show_progress=False):
    extra = {}
    if hp["tgt_lang"] is not None:
        extra["forced_bos_token_id"] = tokenizer.convert_tokens_to_ids(hp["tgt_lang"])

    order = sorted(range(len(sentences)), key=lambda i: len(sentences[i]), reverse=True)
    outputs = [None] * len(sentences)
    batch_size = GENERATION["batch_size"]
    batches = range(0, len(order), batch_size)
    for start in tqdm(batches, disable=not show_progress, leave=False):
        idx = order[start:start + batch_size]
        enc = tokenizer([sentences[i] for i in idx], return_tensors="pt", padding=True,
                        truncation=True, max_length=GENERATION["max_input_tokens"]).to(DEVICE)
        generated = model.generate(**enc, num_beams=GENERATION["num_beams"],
                                   max_new_tokens=GENERATION["max_new_tokens"], **extra)
        unk_in = (enc["input_ids"] == tokenizer.unk_token_id).sum(dim=1)
        unk_out = (generated == tokenizer.unk_token_id).sum(dim=1)
        has_unk = (unk_out > unk_in).tolist()
        texts = tokenizer.batch_decode(generated, skip_special_tokens=True)
        for i, text, unk in zip(idx, texts, has_unk):
            outputs[i] = (text, unk)
    return outputs


# Translates whole tweets: split into sentences, translate unique sentences, join back.
def translate_tweets(texts, tokenizer, model, hp, show_progress=False):
    parts = [split_tweet(text) for text in texts]
    unique = list(dict.fromkeys(s for sentences, _ in parts for s in sentences))
    translated = dict(zip(unique, translate_sentences(unique, tokenizer, model, hp, show_progress)))
    texts_es = [join_tweet([translated[s][0] for s in sentences], kept) for sentences, kept in parts]
    unknown = [any(translated[s][1] for s in sentences) for sentences, _ in parts]
    return texts_es, unknown


def free_memory():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## 6. Controles de calidad

Marcan los tweets que conviene revisar:

- `vacio`: la traducción quedó vacía.
- `sin_traducir`: la traducción es igual al original.
- `longitud`: la traducción mide menos de la mitad o más del doble que el original (solo tweets de 30 caracteres o más).
- `usuarios`: cambió el número de `@USER`.
- `repeticion`: una palabra se repite 4 veces o más seguidas, señal de que el modelo entró en un bucle.
- `desconocido`: la traducción tiene más tokens desconocidos que el original, así que se perdió un carácter (por ejemplo, `Únete` queda como `nete`). Los desconocidos que ya venían en el original, como los emojis dañados de HumAID, no cuentan.

In [ ]:
REPEAT_RE = re.compile(r"\b(\w+)(?:\s+\1\b){3,}", re.IGNORECASE)
FLAGS = ["vacio", "sin_traducir", "longitud", "usuarios", "repeticion", "desconocido"]


def quality_flags(text_en, text_es):
    ratio = len(text_es) / max(len(text_en), 1)
    return {
        "vacio": not text_es.strip(),
        "sin_traducir": text_es.strip().lower() == text_en.strip().lower(),
        "longitud": len(text_en) >= 30 and not 0.5 <= ratio <= 2.0,
        "usuarios": text_en.count("@USER") != text_es.count("@USER"),
        "repeticion": bool(REPEAT_RE.search(text_es)) and not REPEAT_RE.search(text_en),
    }


def add_flags(df):
    df = df.reset_index(drop=True)
    df["text_es"] = df["text_es"].fillna("")
    flags = pd.DataFrame([quality_flags(en, es) for en, es in zip(df[TEXT_COL], df["text_es"])])
    df = pd.concat([df, flags], axis=1)
    df["desconocido"] = df["desconocido"].astype(bool)
    df["alguna_alerta"] = df[FLAGS].any(axis=1)
    return df


def flag_summary(df):
    return {flag: round(float(df[flag].mean()) * 100, 2) for flag in FLAGS + ["alguna_alerta"]}


def show_samples(df, n=10, seed=SEED):
    for _, row in df.sample(n=min(n, len(df)), random_state=seed).iterrows():
        print(f"[{row['label']}]")
        print("EN:", row[TEXT_COL])
        print("ES:", row["text_es"])
        print()

## 7. Piloto: comparar traductores

Toma 100 tweets de train por clase (1.000 en total), los traduce con los tres modelos y mide el tiempo. Las traducciones del piloto solo se usan en memoria para comparar; en `results/traduccion/` quedan los números.

In [ ]:
pilot = (data["train"].sample(frac=1, random_state=SEED)
         .groupby("label").head(PILOT_PER_CLASS).reset_index(drop=True))
print(f"Piloto: {len(pilot):,} tweets")

pilot_rows = []
pilot_outputs = {}
for name, hp in TRANSLATORS.items():
    set_all_seeds(SEED)
    tokenizer, model = load_translator(hp)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    texts_es, unknown = translate_tweets(pilot[TEXT_COL].tolist(), tokenizer, model, hp, show_progress=True)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    seconds = time.time() - start

    result = add_flags(pilot.assign(text_es=texts_es, desconocido=unknown))
    pilot_outputs[name] = result
    pilot_rows.append({
        "traductor": name,
        "modelo": hp["model_name"],
        "segundos": round(seconds, 1),
        "tweets_por_segundo": round(len(pilot) / seconds, 1),
        "minutos_estimados_todo": round(total_tweets / (len(pilot) / seconds) / 60, 1),
        **flag_summary(result),
    })
    print(pilot_rows[-1])
    del model, tokenizer
    free_memory()

pilot_table = pd.DataFrame(pilot_rows)
pilot_table.to_csv(RESULTS_DIR / "piloto_traduccion.csv", index=False)
pilot_table

### Revisión manual del piloto

Muestra los mismos 30 tweets con los tres traductores. Hay que revisar sobre todo que no se pierda información, que los nombres de lugares no se traduzcan y que el sentido de la clase se mantenga.

In [ ]:
review_ids = pilot.sample(n=30, random_state=SEED)["tweet_id"]
by_name = {name: df.set_index("tweet_id") for name, df in pilot_outputs.items()}
for tweet_id in review_ids:
    first = next(iter(by_name.values())).loc[tweet_id]
    print(f"[{first['label']}] {tweet_id}")
    print("EN:      ", first[TEXT_COL])
    for name, df in by_name.items():
        print(f"{name:9s}:", df.loc[tweet_id, "text_es"])
    print()

## 8. Traducción completa

Resultado del piloto en una Tesla T4 (1.000 tweets):

| Traductor | Tweets/s | Tiempo estimado (todo) | Alguna alerta | `desconocido` |
|---|---|---|---|---|
| opus_big | 36,4 | 35 min | 3,8 % | 3,3 % |
| opus_base | 21,0 | 60 min | 0,5 % | 0,0 % |
| nllb | 10,8 | 117 min | 1,1 % | 0,3 % |

Se usa **opus_big**: es el más rápido y en la revisión manual fue el más natural. NLLB se descarta porque omite partes del tweet y llegó a inventar texto ("ymmfire ymm yeg" quedó como "¿Qué hay de nuevo?"). El único problema de opus_big son los caracteres perdidos (`desconocido`), así que esos tweets se traducen de nuevo con **opus_base**, que no tiene ese problema.

La traducción se guarda en bloques de 5.000 tweets: si la sesión se corta, al volver a ejecutar la celda continúa desde el último bloque.

In [ ]:
TRANSLATOR = "opus_big"
FALLBACK = "opus_base"

In [ ]:
def translate_split(name, df, tokenizer, model, hp):
    cache_path = CACHE_DIR / f"{hp['name']}_{name}.csv"
    done_ids = set(pd.read_csv(cache_path, dtype={"tweet_id": str})["tweet_id"]) if cache_path.exists() else set()
    pending = df[~df["tweet_id"].isin(done_ids)]
    print(f"{name}: {len(done_ids):,} ya traducidos, {len(pending):,} pendientes")

    for start in tqdm(range(0, len(pending), CHUNK_SIZE), desc=name):
        chunk = pending.iloc[start:start + CHUNK_SIZE]
        texts_es, unknown = translate_tweets(chunk[TEXT_COL].tolist(), tokenizer, model, hp)
        pd.DataFrame({"tweet_id": chunk["tweet_id"], "text_es": texts_es, "desconocido": unknown}).to_csv(
            cache_path, mode="a", header=not cache_path.exists(), index=False
        )

    translated = pd.read_csv(cache_path, dtype={"tweet_id": str}).drop_duplicates("tweet_id", keep="last")
    return df.merge(translated, on="tweet_id", how="left", validate="one_to_one")


hp = TRANSLATORS[TRANSLATOR]
set_all_seeds(SEED)
tokenizer, model = load_translator(hp)
start_time = time.time()
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

translated = {name: translate_split(name, df, tokenizer, model, hp) for name, df in data.items()}
del model, tokenizer
free_memory()

### Respaldo para caracteres perdidos

Los tweets marcados como `desconocido` se traducen de nuevo con el traductor de respaldo. La columna `traductor` indica cuál quedó en cada tweet.

In [ ]:
fallback_hp = TRANSLATORS[FALLBACK]
tokenizer, model = load_translator(fallback_hp)
for name, df in translated.items():
    df["desconocido"] = df["desconocido"].astype(bool)
    df["traductor"] = TRANSLATOR
    lost = df["desconocido"].copy()
    if lost.any():
        texts_es, unknown = translate_tweets(df.loc[lost, TEXT_COL].tolist(), tokenizer, model, fallback_hp)
        df.loc[lost, "text_es"] = texts_es
        df.loc[lost, "desconocido"] = unknown
        df.loc[lost, "traductor"] = FALLBACK
    print(f"{name}: {int(lost.sum()):,} tweets traducidos de nuevo con {FALLBACK}")
    translated[name] = add_flags(df)
    translated[name].drop(columns=FLAGS + ["alguna_alerta"]).to_csv(OUT_DIR / f"{name}.csv", index=False)

del model, tokenizer
free_memory()
minutes = round((time.time() - start_time) / 60, 2)
print(f"Tiempo total: {minutes} min")

## 9. Reporte de calidad

In [ ]:
summary_rows = []
for name, df in translated.items():
    summary_rows.append({"conjunto": name, "tweets": len(df), "sin_texto": int(df["text_es"].isna().sum()),
                         "con_respaldo": int((df["traductor"] == FALLBACK).sum()), **flag_summary(df)})
summary = pd.DataFrame(summary_rows)

alerts = pd.concat([df.loc[df["alguna_alerta"], ["tweet_id"] + FLAGS].assign(conjunto=name)
                    for name, df in translated.items()])
alerts.to_csv(RESULTS_DIR / "traduccion_alertas.csv", index=False)

report = {
    "traductor": hp["model_name"],
    "respaldo": fallback_hp["model_name"],
    "generacion": GENERATION,
    "tiempo_min": minutes,
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "gpu_memoria_max_usada_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE.type == "cuda" else None,
    "por_conjunto": summary_rows,
}
(RESULTS_DIR / "traduccion.json").write_text(json.dumps(report, indent=2, ensure_ascii=False, default=str))
summary

### Revisión manual

10 tweets al azar de cada conjunto y 10 de los marcados con alguna alerta.

In [ ]:
for name in ["train", "val", "test"]:
    print(f"{'=' * 20} {name} {'=' * 20}")
    show_samples(translated[name])

flagged = pd.concat([df[df["alguna_alerta"]] for df in translated.values()])
print(f"{'=' * 20} con alertas ({len(flagged):,}) {'=' * 20}")
show_samples(flagged)

## 10. Exportar

In [ ]:
if Path("/kaggle").exists():
    shutil.make_archive("/kaggle/working/humaid_es", "zip", OUT_DIR)
    shutil.make_archive("/kaggle/working/resultados_traduccion", "zip", RESULTS_DIR)
    print("Traducción en /kaggle/working/humaid_es.zip y reporte en /kaggle/working/resultados_traduccion.zip")